# Li Double-Transmon Coupler Custom-Circuit Analysis

This notebook reconstructs the Eq. (1) double-transmon-coupler (DTC) Hamiltonian from Li et al., *Realization of High-Fidelity CZ Gate Based on a Double-Transmon Coupler* (Phys. Rev. X 14, 041050, 2024), using the `ScQubitsMimic` custom-circuit pipeline.

The model keeps the four transmon nodes `Q1`, `Q2`, `C3`, and `C4`, the additional coupler junction `JJ5`, and the Table I capacitance matrix. Resonators and control lines are excluded, matching the paper's Eq. (1) approximation.

In [ ]:
using ScQubitsMimic
using CairoMakie
using Statistics
using Printf

const ELEMENTARY_CHARGE = 1.602176634e-19
const PLANCK_CONSTANT = 6.62607015e-34
const FLUX_QUANTUM = PLANCK_CONSTANT / (2 * ELEMENTARY_CHARGE)
const JJ_EC_NEGLIGIBLE = 1.0e18

physical_ec_from_capacitance_ff(C_ff) = ELEMENTARY_CHARGE^2 / (2 * C_ff * 1e-15 * PLANCK_CONSTANT) / 1e9
branch_ec_from_capacitance_ff(C_ff) = physical_ec_from_capacitance_ff(C_ff) / 4
ej_from_critical_current_na(Ic_nA) = FLUX_QUANTUM * Ic_nA * 1e-9 / (2π * PLANCK_CONSTANT) / 1e9
flux_bias_to_rad(bias) = 2π * bias

const IDLE_BIAS = 0.309
const DTC_FLUX_PARAM = :Φ1
const DEFAULT_NCUT = 4
const DIRECT_SANITY_NCUT = 2
const DEFAULT_HIERARCHY = [[1], [2], [[3], [4]]]
const DEFAULT_TRUNC_DIMS = [5, 5, [16, [5, 5]]]
const DIRECT_SANITY_TRUNC_DIMS = [5, 5, [25, [5, 5]]]
const DEFAULT_EVALS_COUNT = 24
const DEFAULT_FLUX_BIAS_GRID = sort!(unique(vcat(
    collect(range(0.25, IDLE_BIAS; length=8)),
    collect(range(IDLE_BIAS, 0.50; length=24)),
)))

@assert length(DEFAULT_FLUX_BIAS_GRID) == 31
@assert IDLE_BIAS in DEFAULT_FLUX_BIAS_GRID

const C11_FF = 91.86
const C22_FF = 91.79
const C33_FF = 110.27
const C44_FF = 106.36
const C12_FF = 0.04
const C13_FF = 5.73
const C14_FF = 0.17
const C23_FF = 0.26
const C24_FF = 5.77
const C34_FF = 1.73

const Q1_EJ_GHz = ej_from_critical_current_na(26.13)
const Q2_EJ_GHz = ej_from_critical_current_na(31.93)
const C3_EJ_GHz = ej_from_critical_current_na(47.73)
const C4_EJ_GHz = ej_from_critical_current_na(47.68)
const JJ5_EJ_GHz = ej_from_critical_current_na(10.32)

const Q1_GROUND_BRANCH_EC = branch_ec_from_capacitance_ff(C11_FF)
const Q2_GROUND_BRANCH_EC = branch_ec_from_capacitance_ff(C22_FF)
const C3_GROUND_BRANCH_EC = branch_ec_from_capacitance_ff(C33_FF)
const C4_GROUND_BRANCH_EC = branch_ec_from_capacitance_ff(C44_FF)
const Q1_Q2_BRANCH_EC = branch_ec_from_capacitance_ff(C12_FF)
const Q1_C3_BRANCH_EC = branch_ec_from_capacitance_ff(C13_FF)
const Q1_C4_BRANCH_EC = branch_ec_from_capacitance_ff(C14_FF)
const Q2_C3_BRANCH_EC = branch_ec_from_capacitance_ff(C23_FF)
const Q2_C4_BRANCH_EC = branch_ec_from_capacitance_ff(C24_FF)
const C3_C4_BRANCH_EC = branch_ec_from_capacitance_ff(C34_FF)

const GROUND_LABEL = (1, 1, 1)
const Q1_LABEL = (2, 1, 1)
const Q2_LABEL = (1, 2, 1)
const Q1Q2_LABEL = (2, 2, 1)

const LI_DTC_CIRCUIT = """
branches:
  - [JJ, 0, 1, EJ=$(Q1_EJ_GHz), EC=$(JJ_EC_NEGLIGIBLE)]
  - [JJ, 0, 2, EJ=$(Q2_EJ_GHz), EC=$(JJ_EC_NEGLIGIBLE)]
  - [JJ, 0, 3, EJ=$(C3_EJ_GHz), EC=$(JJ_EC_NEGLIGIBLE)]
  - [JJ, 0, 4, EJ=$(C4_EJ_GHz), EC=$(JJ_EC_NEGLIGIBLE)]
  - [JJ, 4, 3, EJ=$(JJ5_EJ_GHz), EC=$(JJ_EC_NEGLIGIBLE)]
  - [C, 1, 0, EC=$(Q1_GROUND_BRANCH_EC)]
  - [C, 2, 0, EC=$(Q2_GROUND_BRANCH_EC)]
  - [C, 3, 0, EC=$(C3_GROUND_BRANCH_EC)]
  - [C, 4, 0, EC=$(C4_GROUND_BRANCH_EC)]
  - [C, 1, 2, EC=$(Q1_Q2_BRANCH_EC)]
  - [C, 1, 3, EC=$(Q1_C3_BRANCH_EC)]
  - [C, 1, 4, EC=$(Q1_C4_BRANCH_EC)]
  - [C, 2, 3, EC=$(Q2_C3_BRANCH_EC)]
  - [C, 2, 4, EC=$(Q2_C4_BRANCH_EC)]
  - [C, 3, 4, EC=$(C3_C4_BRANCH_EC)]
"""

function add_capacitance_ff!(C, node_i, node_j, C_ff)
    cap = 1 / (8 * branch_ec_from_capacitance_ff(C_ff))
    if node_i != 0 && node_j != 0
        C[node_i, node_i] += cap
        C[node_j, node_j] += cap
        C[node_i, node_j] -= cap
        C[node_j, node_i] -= cap
    elseif node_i == 0 && node_j != 0
        C[node_j, node_j] += cap
    elseif node_j == 0 && node_i != 0
        C[node_i, node_i] += cap
    end
    return C
end

function expected_capacitance_matrix_from_table_i()
    C = zeros(Float64, 4, 4)
    add_capacitance_ff!(C, 1, 0, C11_FF)
    add_capacitance_ff!(C, 2, 0, C22_FF)
    add_capacitance_ff!(C, 3, 0, C33_FF)
    add_capacitance_ff!(C, 4, 0, C44_FF)
    add_capacitance_ff!(C, 1, 2, C12_FF)
    add_capacitance_ff!(C, 1, 3, C13_FF)
    add_capacitance_ff!(C, 1, 4, C14_FF)
    add_capacitance_ff!(C, 2, 3, C23_FF)
    add_capacitance_ff!(C, 2, 4, C24_FF)
    add_capacitance_ff!(C, 3, 4, C34_FF)
    return C
end

function build_li_dtc_circuit(; flux_bias=IDLE_BIAS, ncut=DEFAULT_NCUT)
    circ = Circuit(LI_DTC_CIRCUIT; ncut=ncut)
    set_param!(circ, DTC_FLUX_PARAM, flux_bias_to_rad(flux_bias))
    return circ
end

function build_hierarchical_hs(; flux_bias=IDLE_BIAS,
                               ncut=DEFAULT_NCUT,
                               trunc_dims=DEFAULT_TRUNC_DIMS)
    return hierarchical_diag(build_li_dtc_circuit(; flux_bias=flux_bias, ncut=ncut);
        system_hierarchy=DEFAULT_HIERARCHY,
        subsystem_trunc_dims=trunc_dims)
end

function build_configured_hierarchical_circuit(; flux_bias=IDLE_BIAS,
                                               ncut=DEFAULT_NCUT,
                                               trunc_dims=DEFAULT_TRUNC_DIMS)
    circ = build_li_dtc_circuit(; flux_bias=flux_bias, ncut=ncut)
    configure!(circ;
        system_hierarchy=DEFAULT_HIERARCHY,
        subsystem_trunc_dims=trunc_dims)
    return circ
end

function subsystem_transition_freq(subsys)
    vals = eigenvals(subsys; evals_count=2)
    return vals[2] - vals[1]
end

subsystem_w01s(hs) = (
    q1 = subsystem_transition_freq(hs.subsystems[1]),
    q2 = subsystem_transition_freq(hs.subsystems[2]),
    dtc = subsystem_transition_freq(hs.subsystems[3]),
)

function lookup_energy(lookup, bare_label::NTuple{3, Int})
    idx = get(lookup.bare_to_dressed, bare_label, nothing)
    idx === nothing && error("Bare label $(bare_label) missing in sweep lookup")
    return lookup.dressed_evals[idx]
end

function zz_from_lookup(lookup)
    E000 = lookup_energy(lookup, GROUND_LABEL)
    E100 = lookup_energy(lookup, Q1_LABEL)
    E010 = lookup_energy(lookup, Q2_LABEL)
    E110 = lookup_energy(lookup, Q1Q2_LABEL)
    return E110 - E100 - E010 + E000
end

function parameter_sweep_li_dtc(flux_bias_vals=DEFAULT_FLUX_BIAS_GRID;
                                trunc_dims=DEFAULT_TRUNC_DIMS,
                                ncut=DEFAULT_NCUT,
                                evals_count=DEFAULT_EVALS_COUNT)
    circ = build_configured_hierarchical_circuit(; flux_bias=first(flux_bias_vals),
        trunc_dims=trunc_dims, ncut=ncut)

    sweep = ParameterSweep(circ._hilbert_space,
        [DTC_FLUX_PARAM => collect(Float64, flux_bias_vals)],
        (sweep::ParameterSweep, flux_bias) -> begin
            set_param!(circ, DTC_FLUX_PARAM, flux_bias_to_rad(flux_bias))
            sweep.hilbertspace = circ._hilbert_space
        end;
        evals_count=evals_count,
        subsys_update_info=Dict(DTC_FLUX_PARAM => [3]),
        ignore_low_overlap=true,
        store_lookups=true)

    relative_levels = sweep.dressed_evals .- sweep.dressed_evals[:, 1]
    zz_values = [zz_from_lookup(lookup) for lookup in sweep.lookups]
    return sweep, relative_levels, zz_values
end

circ1 = build_li_dtc_circuit()


## Circuit Construction

The custom circuit uses the paper's node order `1=Q1`, `2=Q2`, `3=C3`, `4=C4`. Branch 5 is `JJ5` directed from node 4 to node 3, so the symbolic loop term appears as `cos(Φ1 + φ3 - φ4)`, equivalent to the paper's `cos(φ4 - φ3 - φex)` by cosine parity.

In [ ]:
converted_params = (
    EJ_GHz = (
        Q1 = Q1_EJ_GHz,
        Q2 = Q2_EJ_GHz,
        C3 = C3_EJ_GHz,
        C4 = C4_EJ_GHz,
        JJ5 = JJ5_EJ_GHz,
    ),
    branch_EC_GHz = (
        C11 = Q1_GROUND_BRANCH_EC,
        C22 = Q2_GROUND_BRANCH_EC,
        C33 = C3_GROUND_BRANCH_EC,
        C44 = C4_GROUND_BRANCH_EC,
        C12 = Q1_Q2_BRANCH_EC,
        C13 = Q1_C3_BRANCH_EC,
        C14 = Q1_C4_BRANCH_EC,
        C23 = Q2_C3_BRANCH_EC,
        C24 = Q2_C4_BRANCH_EC,
        C34 = C3_C4_BRANCH_EC,
    ),
)

flux_symbols = external_fluxes(circ1)
flux_map = sym_external_fluxes(circ1)
loop_info = flux_map[only(flux_symbols)]

@assert length(flux_symbols) == 1
@assert loop_info.closure_branch == 5
@assert circ1.var_categories.periodic == [1, 2, 3, 4]

(
    converted_params = converted_params,
    external_fluxes = flux_symbols,
    flux_loop_map = flux_map,
    offset_charge_symbols = offset_charges(circ1),
    variable_categories = circ1.var_categories,
)


## Symbolic Lagrangian And Hamiltonian

The symbolic pass is the quantization checkpoint: it verifies the kinetic energy from the capacitance matrix and the Josephson potential terms before numerical diagonalization.

In [ ]:
sym_lagrangian(circ1; vars_type=:node)


In [ ]:
sym_hamiltonian_node(circ1)


The node-basis Hamiltonian matches Li et al. Eq. (1): the charging term is built from the inverse capacitance matrix, the four transmon junctions give `-EJi*cos(φi)`, and branch 5 gives the DTC loop term. The sign convention differs only by the even symmetry of cosine.

In [ ]:
sym_hamiltonian(circ1; return_expr=true)


## Capacitance-Matrix Validation

The branch `EC` values are translated from Table I capacitances and then converted back through the circuit parser. This checks that the graph reproduces the intended capacitance matrix before using it for spectra.

In [ ]:
expected_C = expected_capacitance_matrix_from_table_i()
C_numeric = ScQubitsMimic._build_capacitance_matrix_numeric(circ1)
max_capacitance_error = maximum(abs.(C_numeric .- expected_C))

@assert max_capacitance_error < 1e-12

(
    C_numeric = C_numeric,
    expected_C = expected_C,
    max_abs_error = max_capacitance_error,
)


## Direct vs Hierarchical Spectrum At The Idle Bias

For this sanity check, all four modes use `ncut=2`, making the full direct Hamiltonian small enough to diagonalize exactly. The nested hierarchy keeps Q1 and Q2 separate and treats C3/C4 as one DTC subsystem with its full reduced dimension.

In [ ]:
circ_direct = build_li_dtc_circuit(; ncut=DIRECT_SANITY_NCUT)
direct_evals = eigenvals(circ_direct; evals_count=6)
hs_direct = build_hierarchical_hs(; ncut=DIRECT_SANITY_NCUT, trunc_dims=DIRECT_SANITY_TRUNC_DIMS)
hier_evals = eigenvals(hs_direct; evals_count=6)

direct_relative = direct_evals .- direct_evals[1]
hier_relative = hier_evals .- hier_evals[1]
max_relative_spectrum_error = maximum(abs.(direct_relative .- hier_relative))

@assert max_relative_spectrum_error < 1e-6

(
    direct_relative_levels_GHz = direct_relative,
    hierarchical_relative_levels_GHz = hier_relative,
    max_relative_spectrum_error = max_relative_spectrum_error,
    subsystem_w01_GHz = subsystem_w01s(hs_direct),
)


## Flux Sweep And Dressed ZZ

The sweep scans the reduced flux `φex/2π` across the paper's plotted region and includes the reported idle bias `0.309`. Lookups are stored so Q1-Q2 dressed ZZ can be computed from the dressed energies assigned to `(Q1, Q2, DTC)` bare labels.

In [ ]:
li_sweep, relative_levels_GHz, zz_values_GHz = parameter_sweep_li_dtc()
flux_vals = li_sweep.param_vals[DTC_FLUX_PARAM]
idle_idx = findfirst(==(IDLE_BIAS), flux_vals)
zz_values_kHz = zz_values_GHz .* 1e6

sample_indices = unique([1, idle_idx, length(flux_vals)])
lookup_validation = [
    let lookup = li_sweep.lookups[idx]
        labels_present = Dict(
            GROUND_LABEL => haskey(lookup.bare_to_dressed, GROUND_LABEL),
            Q1_LABEL => haskey(lookup.bare_to_dressed, Q1_LABEL),
            Q2_LABEL => haskey(lookup.bare_to_dressed, Q2_LABEL),
            Q1Q2_LABEL => haskey(lookup.bare_to_dressed, Q1Q2_LABEL),
        )
        @assert all(values(labels_present))
        (
            grid_index = idx,
            flux_bias = flux_vals[idx],
            labels_present = labels_present,
            zz_kHz = zz_values_kHz[idx],
        )
    end
    for idx in sample_indices
]

sweep_summary = (
    param_order = li_sweep.param_order,
    grid_points = length(flux_vals),
    flux_window = extrema(flux_vals),
    idle_index = idle_idx,
    idle_relative_levels_GHz = relative_levels_GHz[idle_idx, 1:6],
    zz_window_kHz = extrema(zz_values_kHz),
    idle_zz_kHz = zz_values_kHz[idle_idx],
    lookup_validation = lookup_validation,
)

sweep_summary


In [ ]:
fig_levels = Figure(size=(920, 460))
ax_levels = Axis(fig_levels[1, 1];
    xlabel="Reduced flux φex / 2π",
    ylabel="Dressed transition from ground (GHz)",
    title="Li DTC low-energy dressed spectrum")

for level in 2:6
    lines!(ax_levels, flux_vals, relative_levels_GHz[:, level];
        linewidth=2.2, label="E$(level - 1) - E0")
end
vlines!(ax_levels, [IDLE_BIAS]; color=:gray45, linestyle=:dash, linewidth=1.8, label="idle")
axislegend(ax_levels; position=:lt)
fig_levels


In [ ]:
fig_zz = Figure(size=(920, 420))
ax_zz = Axis(fig_zz[1, 1];
    xlabel="Reduced flux φex / 2π",
    ylabel="ζ / 2π (kHz)",
    title="Q1-Q2 exact dressed ZZ from lookup energies")

lines!(ax_zz, flux_vals, zz_values_kHz; color=:darkorange, linewidth=2.8, label="ZZ")
scatter!(ax_zz, flux_vals, zz_values_kHz; color=:darkorange, markersize=7)
hlines!(ax_zz, [0.0]; color=:gray55, linestyle=:dash, linewidth=1.4)
vlines!(ax_zz, [IDLE_BIAS]; color=:gray35, linestyle=:dash, linewidth=1.8, label="idle")
axislegend(ax_zz; position=:rt)
fig_zz


## Notes

The sweep is intentionally modest for interactive execution. Increase `DEFAULT_NCUT`, `DEFAULT_TRUNC_DIMS`, and `DEFAULT_EVALS_COUNT` together for denser convergence studies or paper-figure reproduction.